In [ ]:
import os
import torch
import pandas as pd
import scanpy as sc
import skmisc
import numpy as np
import torch_geometric
import time
from Model.VGAE import VGAE, train_vgae
from Model.BGRL import BGRL, train_bgrl, get_embedding_bgrl
from Model.DGI import DGI, train_dgi, get_embedding_dgi
from Model.GAE import GAE, train_gae
from Model.GATE import GATE, train_gate
from utils import cKD_refine_label, buildGraph, compute_metrics, fix_seed
from sklearn.mixture import BayesianGaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score

In [2]:
# Environment configuration
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
file_fold = r"../dataset"

In [4]:
# Fix random seed for reproducibility
random_seed = 42
fix_seed(random_seed)

## Simulated data

In [60]:
simulation_fold = file_fold + "/simulation"

In [ ]:
# read data 1
rna_df_1 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_RNA.csv")
adt_df_1 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_ADT.csv")
meta_df_1 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_100_metadata.csv")

In [ ]:
# read data 2
rna_df_2 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_RNA.csv")
adt_df_2 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_ADT.csv")
meta_df_2 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_1992_nGenes_1000_metadata.csv")

In [61]:
# read data 3
rna_df_3 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_100_RNA.csv")
adt_df_3 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_100_ADT.csv")
meta_df_3 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_100_metadata.csv")

In [ ]:
# read data 4
rna_df_4 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_1000_RNA.csv")
adt_df_4 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_1000_ADT.csv")
meta_df_4 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_5024_nGenes_1000_metadata.csv")

In [ ]:
# read data 5
rna_df_5 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_RNA.csv")
adt_df_5 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_ADT.csv")
meta_df_5 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_100_metadata.csv")

In [ ]:
# read data 6
rna_df_6 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_RNA.csv")
adt_df_6 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_ADT.csv")
meta_df_6 = pd.read_csv(simulation_fold + r"/Amplify_CBMCs_nCells_10075_nGenes_1000_metadata.csv")

In [62]:
def create_adata(rna_df, adt_df, meta_df):
    adata_rna = sc.AnnData(X = rna_df.values, obs = meta_df, var = pd.DataFrame(index = rna_df.columns), dtype = "float32")
    adata_adt = sc.AnnData(X = adt_df.values, obs = meta_df, var = pd.DataFrame(index = adt_df.columns), dtype = "float32")

    assert (rna_df.index == meta_df.index).all()
    assert (adt_df.index == meta_df.index).all()
    
    adata_rna.var_names_make_unique()
    adata_adt.var_names_make_unique()
    
    adata_rna.obsm['spatial'] = adata_rna.obs[['X','Y']].to_numpy()
    adata_adt.obsm['spatial'] = adata_adt.obs[['X','Y']].to_numpy()

    return adata_rna, adata_adt

In [63]:
#adata_rna_1, adata_adt_1 = create_adata(rna_df = rna_df_1, adt_df = adt_df_1, meta_df = meta_df_1)
#adata_rna_2, adata_adt_2 = create_adata(rna_df = rna_df_2, adt_df = adt_df_2, meta_df = meta_df_2)
adata_rna_3, adata_adt_3 = create_adata(rna_df = rna_df_3, adt_df = adt_df_3, meta_df = meta_df_3)
#adata_rna_4, adata_adt_4 = create_adata(rna_df = rna_df_4, adt_df = adt_df_4, meta_df = meta_df_4)
#adata_rna_5, adata_adt_5 = create_adata(rna_df = rna_df_5, adt_df = adt_df_5, meta_df = meta_df_5)
#adata_rna_6, adata_adt_6 = create_adata(rna_df = rna_df_6, adt_df = adt_df_6, meta_df = meta_df_6)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [ ]:
from utils import clr_normalize_each_cell, pca

def preprocess(adata_rna, adata_adt):
    # RNA
    sc.pp.filter_genes(adata_rna, min_cells=10)
    sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_rna, target_sum=1e4)
    sc.pp.log1p(adata_rna)
    sc.pp.scale(adata_rna)
    
    adata_rna_high =  adata_rna[:, adata_rna.var['highly_variable']]
    adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=adata_adt.n_vars-1)

    # Protein
    adata_adt = clr_normalize_each_cell(adata_adt)
    sc.pp.scale(adata_adt)
    adata_adt.obsm['feat'] = pca(adata_adt, n_comps=adata_adt.n_vars-1)

    return adata_rna, adata_adt

In [65]:
#adata_rna_1, adata_adt_1 = preprocess(adata_rna_1, adata_adt_1)
#adata_rna_2, adata_adt_2 = preprocess(adata_rna_2, adata_adt_2)
adata_rna_3, adata_adt_3 = preprocess(adata_rna_3, adata_adt_3)
#adata_rna_4, adata_adt_4 = preprocess(adata_rna_4, adata_adt_4)
#adata_rna_5, adata_adt_5 = preprocess(adata_rna_5, adata_adt_5)
#adata_rna_6, adata_adt_6 = preprocess(adata_rna_6, adata_adt_6)

In [14]:
print(adata_rna_1.obsm['feat'].shape)
print(adata_rna_2.obsm['feat'].shape)
print(adata_rna_3.obsm['feat'].shape)
print(adata_rna_4.obsm['feat'].shape)
print(adata_rna_5.obsm['feat'].shape)
print(adata_rna_6.obsm['feat'].shape)

(1992, 5)
(1992, 5)
(5024, 5)
(5024, 5)
(10075, 5)
(10075, 5)


In [15]:
print(adata_adt_1.obsm['feat'].shape)
print(adata_adt_2.obsm['feat'].shape)
print(adata_adt_3.obsm['feat'].shape)
print(adata_adt_4.obsm['feat'].shape)
print(adata_adt_5.obsm['feat'].shape)
print(adata_adt_6.obsm['feat'].shape)

(1992, 5)
(1992, 5)
(5024, 5)
(5024, 5)
(10075, 5)
(10075, 5)


In [66]:
# Build graph
#data_1 = buildGraph(adata_rna_1, adata_adt_1, k=10)
#data_2 = buildGraph(adata_rna_2, adata_adt_2, k=10)
data_3 = buildGraph(adata_rna_3, adata_adt_3, k=10)
#data_4 = buildGraph(adata_rna_4, adata_adt_4, k=10)
#data_5 = buildGraph(adata_rna_5, adata_adt_5, k=10)
#data_6 = buildGraph(adata_rna_6, adata_adt_6, k=10)

In [67]:
#print(data_1)
#print(data_2)
print(data_3)
#print(data_4)
#print(data_5)
#print(data_6)

Data(num_nodes=5024, edge_index=[2, 100480], x_omics1=[5024, 5], x_omics2=[5024, 5])


## Real data

### Data R1

In [5]:
import h5py

# Open the file
with h5py.File(file_fold + "/ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5", "r") as f:
    print("Keys:", list(f.keys()))
    
    # Load RNA matrix
    X_rna = np.array(f["X_RNA"])
    
    # Load ATAC matrix
    X_atac = np.array(f["X_ATAC"])
    
    # Load cell names
    cells = [c.decode() if isinstance(c, bytes) else c for c in f["Cell"][:]]
    
    # Load gene names
    genes = [g.decode() if isinstance(g, bytes) else g for g in f["Gene"][:]]
    
    # Load positions (spatial coordinates)
    pos = np.array(f["Pos"])

    # Load labels
    labels = [l.decode() if isinstance(l, bytes) else l for l in f["LayerName"][:]]

Keys: ['Cell', 'Gene', 'LayerName', 'Pos', 'X_ATAC', 'X_RNA']


In [6]:
#Create anndata object for RNA

adata_rna_r1 = sc.AnnData(X=X_rna)
adata_rna_r1.obs_names = cells
adata_rna_r1.var_names = genes
adata_rna_r1.obsm["spatial"] = pos
adata_rna_r1.obs["modality"] = "RNA"
adata_rna_r1.obs["label"] = labels

/tmp/ipykernel_125469/3305888143.py:3: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_rna_r1 = sc.AnnData(X=X_rna)


In [7]:
#Create anndata object for ATAC

adata_atac_r1 = sc.AnnData(X=X_atac)
adata_atac_r1.obs_names = cells
adata_atac_r1.obsm["spatial"] = pos
adata_atac_r1.obs["modality"] = "ATAC"
adata_atac_r1.obs["label"] = labels

/tmp/ipykernel_125469/693086368.py:3: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_atac_r1 = sc.AnnData(X=X_atac)


In [8]:
adata_rna_r1.X

array([[-0.1408649 , -0.25902408, -0.0329933 , ..., -0.17913185,
        -0.19040895, -0.1951774 ],
       [-0.16861354, -0.02633325, -0.01485257, ..., -0.17191124,
        -0.23667939, -0.18246482],
       [-0.18041976,  0.05589169, -0.00882955, ..., -0.17132908,
        -0.25573117, -0.18019445],
       ...,
       [-0.22271697, -0.16404475, -0.03974758, ..., -0.2454363 ,
        -0.3037712 , -0.26777905],
       [-0.17255262, -0.08056004, -0.02100811, ..., -0.18386504,
        -0.24008381, -0.19707741],
       [-0.17454453, -0.12228561, -0.02559579, ..., -0.1920227 ,
        -0.24121702, -0.20710994]], dtype=float32)

In [8]:
from SpatialGlue.preprocess import pca

adata_rna_r1.obsm['feat'] = pca(adata_rna_r1, n_comps=adata_atac_r1.n_vars-1)
adata_atac_r1.obsm['feat'] = pca(adata_atac_r1, n_comps=adata_atac_r1.n_vars-1)

In [ ]:
#adata_rna_r1.obsm['feat'] = adata_rna_r1.X
#adata_atac_r1.obsm['feat'] = adata_atac_r1.X

In [9]:
print(adata_rna_r1.obsm['feat'].shape)
print(adata_atac_r1.obsm['feat'].shape)

(9215, 49)
(9215, 49)


In [10]:
data_r1 = buildGraph(adata_rna_r1, adata_atac_r1, k=6)

In [11]:
print(data_r1)

Data(num_nodes=9215, edge_index=[2, 110580], x_omics1=[9215, 49], x_omics2=[9215, 49])


### Data R2

In [86]:
import h5py

# Open the file
f_atac = h5py.File(file_fold + "/MISAR_seq_mouse_E15_brain_data/MISAR_seq_mouse_E15_brain_ATAC_data.h5", "r")
print("Keys:", list(f_atac.keys()))

# Open the file
f_rna = h5py.File(file_fold + "/MISAR_seq_mouse_E15_brain_data/MISAR_seq_mouse_E15_brain_mRNA_data.h5", "r")
print("Keys:", list(f_rna.keys()))

Keys: ['X', 'Y', 'cell', 'peak', 'pos']
Keys: ['X', 'Y', 'cell', 'gene', 'pos']


In [87]:
# Check if f_rna cell, pos, Y is identical with f_atac cell, pos, Y
assert (f_rna['cell'][:] == f_atac['cell'][:]).all()
assert (f_rna['pos'][:] == f_atac['pos'][:]).all()
assert (f_rna['Y'][:] == f_atac['Y'][:]).all()

In [88]:
# Load RNA matrix
X_rna = np.array(f_rna["X"])
    
# Load ATAC matrix
X_atac = np.array(f_atac["X"])
    
# Load cell names
cells = [c.decode() if isinstance(c, bytes) else c for c in f_rna["cell"][:]]
    
# Load gene names
genes = [g.decode() if isinstance(g, bytes) else g for g in f_rna["gene"][:]]

# Load protein names
peaks = [p.decode() if isinstance(p, bytes) else p for p in f_atac['peak'][:]]

# Load positions (spatial coordinates)
pos = np.array(f_rna["pos"])

# Load labels
labels = [l.decode() if isinstance(l, bytes) else l for l in f_rna["Y"][:]]

In [89]:
#Create anndata object for RNA

adata_rna_r2 = sc.AnnData(X=X_rna)
adata_rna_r2.obs_names = cells
adata_rna_r2.var_names = genes
adata_rna_r2.obsm["spatial"] = pos
adata_rna_r2.obs["modality"] = "RNA"
adata_rna_r2.obs["label"] = labels

/tmp/ipykernel_77519/4230971838.py:3: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_rna_r2 = sc.AnnData(X=X_rna)


In [90]:
#Create anndata object for ATAC

adata_atac_r2 = sc.AnnData(X=X_atac)
adata_atac_r2.obs_names = cells
adata_atac_r2.obsm["spatial"] = pos
adata_atac_r2.var_names = peaks
adata_atac_r2.obs["modality"] = "ATAC"
adata_atac_r2.obs["label"] = labels

/tmp/ipykernel_77519/3958626685.py:3: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata_atac_r2 = sc.AnnData(X=X_atac)


In [14]:
print(adata_rna_r2.X.shape)
print(adata_atac_r2.X.shape)

(1949, 2144)
(1949, 47287)


In [22]:
print(adata_rna_r2.X)

[[ 3. 32.  1. ...  1.  3. 14.]
 [ 3. 34.  6. ...  4.  6. 29.]
 [ 5. 32.  2. ...  2.  2. 23.]
 ...
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]
 [ 0.  0.  0. ...  0.  0.  0.]]


In [91]:
from SpatialGlue.preprocess import pca

def preprocess_r2(adata_rna, adata_atac):
    # RNA
    sc.pp.filter_genes(adata_rna, min_cells=10)
    sc.pp.highly_variable_genes(adata_rna, flavor="seurat_v3", n_top_genes=2000)
    sc.pp.normalize_total(adata_rna, target_sum=1e4)
    sc.pp.log1p(adata_rna)
    sc.pp.scale(adata_rna)
    
    adata_rna_high =  adata_rna[:, adata_rna.var['highly_variable']]
    # choose a safe number of PCA components
    n_comps = min(50, adata_atac.n_obs-1, adata_atac.n_vars-1)
    print(f"Using {n_comps} PCA components for RNA and ATAC")
    adata_rna.obsm['feat'] = pca(adata_rna_high, n_comps=n_comps)

    # Epgigenetic
    sc.pp.filter_genes(adata_atac, min_cells=10)
    sc.pp.highly_variable_genes(adata_atac, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_atac, target_sum=1e4)
    sc.pp.log1p(adata_atac)
    sc.pp.scale(adata_atac)

    adata_atac_high =  adata_atac[:, adata_atac.var['highly_variable']]

    adata_atac.obsm['feat'] = pca(adata_atac_high, n_comps=n_comps)

    return adata_rna, adata_atac

In [92]:
preprocess_r2(adata_rna_r2, adata_atac_r2)

Using 50 PCA components for RNA and ATAC


(AnnData object with n_obs × n_vars = 1949 × 2144
     obs: 'modality', 'label'
     var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'hvg', 'log1p'
     obsm: 'spatial', 'feat',
 AnnData object with n_obs × n_vars = 1949 × 47287
     obs: 'modality', 'label'
     var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
     uns: 'hvg', 'log1p'
     obsm: 'spatial', 'feat')

In [93]:
adata_rna_r2.obsm['feat'].shape, adata_atac_r2.obsm['feat'].shape

((1949, 50), (1949, 50))

In [31]:
adata_atac_r2.obsm['feat']

array([[  3.7033477 ,   3.2254694 ,  -2.0909057 , ...,   0.69647   ,
         -0.51136094,   0.22308111],
       [  5.262065  ,   2.979859  ,   1.3067812 , ...,   1.2595049 ,
         -2.5216815 ,   0.13291296],
       [  2.2161183 ,   1.3112347 ,  -4.725282  , ...,   0.57968247,
         -0.39812857,  -0.9597941 ],
       ...,
       [-10.060724  ,  -4.1280947 ,  -2.4002333 , ...,  -0.6895618 ,
          1.1438849 ,   1.1478049 ],
       [-11.916029  ,  -2.7279968 ,   0.11311049, ...,  -0.6837425 ,
         -0.393745  ,   0.8321063 ],
       [ -8.743897  ,  -4.924308  ,  -1.9313706 , ...,  -1.0927794 ,
          1.089068  ,  -0.06309277]], dtype=float32)

In [95]:
data_r2 = buildGraph(adata_rna_r2, adata_atac_r2, k=10)
print(data_r2)

Data(num_nodes=1949, edge_index=[2, 38980], x_omics1=[1949, 50], x_omics2=[1949, 50])


### Data R3

In [7]:
#real data
real_data = sc.read_h5ad(file_fold + r"/adata_all_human_lymph_node_A1.h5ad")

rna_df_r3 = real_data.obsm['RNA_feat']
adt_df_r3 = real_data.obsm['Pro_feat']

adata_rna_r3 = sc.AnnData(X = rna_df_r3, obs = real_data.obs)
adata_adt_r3 = sc.AnnData(X = adt_df_r3, obs = real_data.obs)
adata_rna_r3.obsm['spatial'] = real_data.obsm['spatial']
adata_adt_r3.obsm['spatial'] = real_data.obsm['spatial']

In [16]:
adata_rna_r3.obsm['feat'] = adata_rna_r3.X
adata_adt_r3.obsm['feat'] = adata_adt_r3.X

In [17]:
data_r3 = buildGraph(adata_rna_r3, adata_adt_r3, k=10)

## VGAE

### Data 3

In [68]:
epochs = 1000

model_3 = VGAE(
    dropout = 0.2,
    in_omics1 = data_3.x_omics1.shape[1],
    in_omics2 = data_3.x_omics2.shape[1],
    recon_omics1_dim = data_3.x_omics1.shape[1],
    recon_omics2_dim = data_3.x_omics2.shape[1],
    recon_spatial_dim = data_3.edge_index.shape[1]
    ).to(device)
z = train_vgae(model_3, data = data_3, epochs = epochs, device = device)

Epoch 010 | Total: 13.0614 | Graph: 1.8461 | KLD: 0.0384 | Omics1: 10.0141 | Omics2: 1.1628
Epoch 020 | Total: 11.8798 | Graph: 2.1204 | KLD: 0.3873 | Omics1: 8.2986 | Omics2: 1.0735
Epoch 030 | Total: 10.7865 | Graph: 3.0694 | KLD: 1.1293 | Omics1: 5.6104 | Omics2: 0.9774
Epoch 040 | Total: 9.6073 | Graph: 3.1876 | KLD: 0.6539 | Omics1: 4.8774 | Omics2: 0.8884
Epoch 050 | Total: 8.8662 | Graph: 3.5671 | KLD: 0.6518 | Omics1: 3.9104 | Omics2: 0.7369
Epoch 060 | Total: 8.1160 | Graph: 3.4922 | KLD: 0.7083 | Omics1: 3.2739 | Omics2: 0.6416
Epoch 070 | Total: 7.1557 | Graph: 2.8321 | KLD: 0.6852 | Omics1: 3.0240 | Omics2: 0.6144
Epoch 080 | Total: 6.5960 | Graph: 2.4223 | KLD: 0.7000 | Omics1: 2.8691 | Omics2: 0.6046
Epoch 090 | Total: 6.2493 | Graph: 2.1778 | KLD: 0.7048 | Omics1: 2.7786 | Omics2: 0.5881
Epoch 100 | Total: 5.9642 | Graph: 1.9451 | KLD: 0.7094 | Omics1: 2.7348 | Omics2: 0.5749
Epoch 110 | Total: 5.7755 | Graph: 1.8130 | KLD: 0.7339 | Omics1: 2.6670 | Omics2: 0.5615
Epoch 

In [69]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_3.copy()
num_cluster_3 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=15, random_state=42)

In [84]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 50)

In [85]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4230
NMI: 0.6539
HOM: 0.6381


### Data 4

In [61]:
epochs = 400

model_4 = VGAE(
    dropout = 0.2,
    in_omics1 = data_4.x_omics1.shape[1],
    in_omics2 = data_4.x_omics2.shape[1],
    recon_omics1_dim = data_4.x_omics1.shape[1],
    recon_omics2_dim = data_4.x_omics2.shape[1],
    recon_spatial_dim = data_4.edge_index.shape[1]
    ).to(device)
z = train_vgae(model_4, data = data_4, epochs = epochs, device = device)

Epoch 001 | Total: 94.7300 | Graph: 2.4995 | KLD: 0.0341 | Omics1: 90.9845 | Omics2: 1.2118
Epoch 002 | Total: 94.2700 | Graph: 2.3198 | KLD: 0.0266 | Omics1: 90.7295 | Omics2: 1.1942
Epoch 003 | Total: 93.9081 | Graph: 2.1686 | KLD: 0.0245 | Omics1: 90.5284 | Omics2: 1.1867
Epoch 004 | Total: 93.5716 | Graph: 2.0651 | KLD: 0.0264 | Omics1: 90.2921 | Omics2: 1.1879
Epoch 005 | Total: 93.1986 | Graph: 1.9686 | KLD: 0.0316 | Omics1: 90.0166 | Omics2: 1.1818
Epoch 006 | Total: 92.7562 | Graph: 1.8557 | KLD: 0.0407 | Omics1: 89.6861 | Omics2: 1.1738
Epoch 007 | Total: 92.3045 | Graph: 1.8065 | KLD: 0.0538 | Omics1: 89.2725 | Omics2: 1.1716
Epoch 008 | Total: 91.7657 | Graph: 1.8016 | KLD: 0.0740 | Omics1: 88.7282 | Omics2: 1.1619
Epoch 009 | Total: 91.2628 | Graph: 1.9143 | KLD: 0.1021 | Omics1: 88.0920 | Omics2: 1.1545
Epoch 010 | Total: 90.7480 | Graph: 2.1867 | KLD: 0.1401 | Omics1: 87.2765 | Omics2: 1.1447
Epoch 011 | Total: 90.0322 | Graph: 2.5270 | KLD: 0.1920 | Omics1: 86.1834 | Omi

In [63]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_4.copy()
num_cluster_4 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_4,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [64]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [65]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4487
NMI: 0.6692
HOM: 0.6712


### Data 5

In [26]:
epochs = 500

model_5 = VGAE(
    dropout = 0.2,
    in_omics1 = data_5.x_omics1.shape[1],
    in_omics2 = data_5.x_omics2.shape[1],
    recon_omics1_dim = data_5.x_omics1.shape[1],
    recon_omics2_dim = data_5.x_omics2.shape[1],
    recon_spatial_dim = data_5.edge_index.shape[1]
    ).to(device)
z = train_vgae(model_5, data = data_5, epochs = epochs, device = device)

Epoch 001 | Total: 13.8153 | Graph: 2.2720 | KLD: 0.0110 | Omics1: 10.3093 | Omics2: 1.2231
Epoch 002 | Total: 13.6679 | Graph: 2.1737 | KLD: 0.0092 | Omics1: 10.2738 | Omics2: 1.2112
Epoch 003 | Total: 13.5402 | Graph: 2.0919 | KLD: 0.0086 | Omics1: 10.2370 | Omics2: 1.2027
Epoch 004 | Total: 13.4204 | Graph: 2.0121 | KLD: 0.0090 | Omics1: 10.2041 | Omics2: 1.1953
Epoch 005 | Total: 13.3062 | Graph: 1.9413 | KLD: 0.0104 | Omics1: 10.1675 | Omics2: 1.1871
Epoch 006 | Total: 13.1977 | Graph: 1.8811 | KLD: 0.0129 | Omics1: 10.1218 | Omics2: 1.1819
Epoch 007 | Total: 13.0833 | Graph: 1.8290 | KLD: 0.0165 | Omics1: 10.0614 | Omics2: 1.1765
Epoch 008 | Total: 12.9468 | Graph: 1.7548 | KLD: 0.0220 | Omics1: 9.9947 | Omics2: 1.1752
Epoch 009 | Total: 12.8098 | Graph: 1.6989 | KLD: 0.0297 | Omics1: 9.9124 | Omics2: 1.1688
Epoch 010 | Total: 12.6429 | Graph: 1.6278 | KLD: 0.0400 | Omics1: 9.8081 | Omics2: 1.1670
Epoch 011 | Total: 12.4374 | Graph: 1.5371 | KLD: 0.0540 | Omics1: 9.6851 | Omics2:

In [27]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_5.copy()
num_cluster_5 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_5,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [28]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [29]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3968
NMI: 0.6351
HOM: 0.6176


### Data 6

In [30]:
epochs = 500

model_6 = VGAE(
    dropout = 0.2,
    in_omics1 = data_6.x_omics1.shape[1],
    in_omics2 = data_6.x_omics2.shape[1],
    recon_omics1_dim = data_6.x_omics1.shape[1],
    recon_omics2_dim = data_6.x_omics2.shape[1],
    recon_spatial_dim = data_6.edge_index.shape[1]
    ).to(device)
z = train_vgae(model_6, data = data_6, epochs = epochs, device = device)

Epoch 001 | Total: 94.7175 | Graph: 2.7147 | KLD: 0.0612 | Omics1: 90.7300 | Omics2: 1.2115
Epoch 002 | Total: 93.8357 | Graph: 2.3730 | KLD: 0.0558 | Omics1: 90.2095 | Omics2: 1.1974
Epoch 003 | Total: 93.1465 | Graph: 2.1738 | KLD: 0.0585 | Omics1: 89.7248 | Omics2: 1.1894
Epoch 004 | Total: 92.4924 | Graph: 2.0810 | KLD: 0.0698 | Omics1: 89.1600 | Omics2: 1.1816
Epoch 005 | Total: 91.8576 | Graph: 2.1744 | KLD: 0.0878 | Omics1: 88.4236 | Omics2: 1.1718
Epoch 006 | Total: 91.1940 | Graph: 2.3757 | KLD: 0.1137 | Omics1: 87.5428 | Omics2: 1.1618
Epoch 007 | Total: 90.6799 | Graph: 2.8526 | KLD: 0.1521 | Omics1: 86.5234 | Omics2: 1.1517
Epoch 008 | Total: 90.1937 | Graph: 3.5463 | KLD: 0.1980 | Omics1: 85.3103 | Omics2: 1.1391
Epoch 009 | Total: 89.5990 | Graph: 4.4659 | KLD: 0.2651 | Omics1: 83.7462 | Omics2: 1.1218
Epoch 010 | Total: 89.1828 | Graph: 5.7985 | KLD: 0.3511 | Omics1: 81.9229 | Omics2: 1.1104
Epoch 011 | Total: 88.6540 | Graph: 7.4184 | KLD: 0.4692 | Omics1: 79.6686 | Omi

In [31]:
# Continue training
z = train_vgae(model_6, data = data_6, epochs = 100, device = device)

Epoch 001 | Total: 14.9719 | Graph: 4.0166 | KLD: 1.4667 | Omics1: 9.1194 | Omics2: 0.3692
Epoch 002 | Total: 22.4189 | Graph: 4.7347 | KLD: 1.5425 | Omics1: 15.7595 | Omics2: 0.3821
Epoch 003 | Total: 15.2464 | Graph: 3.8689 | KLD: 1.4637 | Omics1: 9.5457 | Omics2: 0.3681
Epoch 004 | Total: 16.9285 | Graph: 4.3951 | KLD: 1.4341 | Omics1: 10.7199 | Omics2: 0.3794
Epoch 005 | Total: 19.4536 | Graph: 4.9837 | KLD: 1.4351 | Omics1: 12.6501 | Omics2: 0.3848
Epoch 006 | Total: 18.5798 | Graph: 4.8088 | KLD: 1.4273 | Omics1: 11.9640 | Omics2: 0.3797
Epoch 007 | Total: 16.3440 | Graph: 4.2623 | KLD: 1.4265 | Omics1: 10.2806 | Omics2: 0.3746
Epoch 008 | Total: 15.3690 | Graph: 3.9045 | KLD: 1.4368 | Omics1: 9.6513 | Omics2: 0.3764
Epoch 009 | Total: 15.3921 | Graph: 3.7355 | KLD: 1.4644 | Omics1: 9.8182 | Omics2: 0.3741
Epoch 010 | Total: 16.2490 | Graph: 3.8464 | KLD: 1.4887 | Omics1: 10.5387 | Omics2: 0.3752
Epoch 011 | Total: 16.6655 | Graph: 3.8390 | KLD: 1.4933 | Omics1: 10.9566 | Omics2:

In [32]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_6.copy()
num_cluster_6 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_6,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [33]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [34]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4491
NMI: 0.6728
HOM: 0.6676


### Data R1

In [12]:
import time

total_start = time.time()

In [ ]:
vgae_start = time.time()

vgae1 = VGAE(
    dropout = 0.2,
    in_omics1 = data_r1.x_omics1.shape[1],
    in_omics2 = data_r1.x_omics2.shape[1],
    recon_omics1_dim = data_r1.x_omics1.shape[1],
    recon_omics2_dim = data_r1.x_omics2.shape[1],
    recon_spatial_dim = data_r1.edge_index.shape[1]
    ).to(device)   

z = train_vgae(vgae1, data = data_r1, epochs = 200, device = device)

vgae_end = time.time()
vgae_time = vgae_end - vgae_start
print(f"VGAE time: {vgae_time:.2f}s")

In [17]:
z = train_vgae(vgae1, data = data_r1, epochs = 100, device = device)

Epoch 001 | Total: 5.6150 | Graph: 0.7511 | KLD: 0.4274 | Omics1: 3.5101 | Omics2: 0.9264
Epoch 002 | Total: 5.6588 | Graph: 0.7589 | KLD: 0.4406 | Omics1: 3.5326 | Omics2: 0.9267
Epoch 003 | Total: 5.6153 | Graph: 0.7531 | KLD: 0.4162 | Omics1: 3.5193 | Omics2: 0.9267
Epoch 004 | Total: 5.6505 | Graph: 0.7769 | KLD: 0.4070 | Omics1: 3.5379 | Omics2: 0.9288
Epoch 005 | Total: 5.6361 | Graph: 0.7717 | KLD: 0.4062 | Omics1: 3.5288 | Omics2: 0.9293
Epoch 006 | Total: 5.6093 | Graph: 0.7475 | KLD: 0.4089 | Omics1: 3.5250 | Omics2: 0.9280
Epoch 007 | Total: 5.5921 | Graph: 0.7189 | KLD: 0.4213 | Omics1: 3.5248 | Omics2: 0.9271
Epoch 008 | Total: 5.5830 | Graph: 0.6984 | KLD: 0.4339 | Omics1: 3.5243 | Omics2: 0.9264
Epoch 009 | Total: 5.6015 | Graph: 0.7131 | KLD: 0.4442 | Omics1: 3.5174 | Omics2: 0.9268
Epoch 010 | Total: 5.6170 | Graph: 0.7284 | KLD: 0.4486 | Omics1: 3.5140 | Omics2: 0.9260
Epoch 011 | Total: 5.6032 | Graph: 0.7229 | KLD: 0.4481 | Omics1: 3.5086 | Omics2: 0.9237
Epoch 012 

In [17]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r1.copy()
num_cluster_r1 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r1,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )

# Clustering
cluster_start = time.time()

bayes.fit(df_z)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=9, random_state=42)

In [18]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)

cluster_end = time.time()
cluster_time = cluster_end - cluster_start
print(f"Clustering time: {cluster_time:.2f}s")

refine_start = time.time()
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)
refine_end = time.time()

refine_time = refine_end - refine_start
total_time = refine_end - total_start
print(f"Refinement time: {refine_time:.2f}s")
print(f"Total time: {total_time:.2f}s")

Clustering time: 11.64s
Refinement time: 0.18s
Total time: 925.83s


In [19]:
dataset = "ATAC_RNA_Seq_MouseBrain"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2445
NMI: 0.4361
HOM: 0.3699


### Data R2

In [96]:
vgae_start = time.time()

vgae2 = VGAE(
    dropout = 0.2,
    in_omics1 = data_r2.x_omics1.shape[1],
    in_omics2 = data_r2.x_omics2.shape[1],
    recon_omics1_dim = data_r2.x_omics1.shape[1],
    recon_omics2_dim = data_r2.x_omics2.shape[1],
    recon_spatial_dim = data_r2.edge_index.shape[1]
    ).to(device)   

z = train_vgae(vgae2, data = data_r2, epochs = 1000, device = device)

vgae_end = time.time()
vgae_time = vgae_end - vgae_start
print(f"VGAE time: {vgae_time:.2f}s")

Epoch 010 | Total: 20.7529 | Graph: 1.3990 | KLD: 0.1204 | Omics1: 6.6783 | Omics2: 12.5552
Epoch 020 | Total: 20.3886 | Graph: 1.1122 | KLD: 0.6014 | Omics1: 6.5225 | Omics2: 12.1526
Epoch 030 | Total: 19.9151 | Graph: 1.7822 | KLD: 0.7977 | Omics1: 6.3233 | Omics2: 11.0119
Epoch 040 | Total: 18.7899 | Graph: 1.8302 | KLD: 0.9128 | Omics1: 6.0477 | Omics2: 9.9991
Epoch 050 | Total: 18.2864 | Graph: 2.5589 | KLD: 1.0580 | Omics1: 5.6819 | Omics2: 8.9876
Epoch 060 | Total: 17.3647 | Graph: 2.8207 | KLD: 1.1886 | Omics1: 5.2160 | Omics2: 8.1395
Epoch 070 | Total: 16.3264 | Graph: 2.6773 | KLD: 1.1234 | Omics1: 4.9305 | Omics2: 7.5951
Epoch 080 | Total: 15.4738 | Graph: 2.5075 | KLD: 1.0984 | Omics1: 4.7732 | Omics2: 7.0948
Epoch 090 | Total: 14.5617 | Graph: 2.1636 | KLD: 1.1221 | Omics1: 4.6496 | Omics2: 6.6264
Epoch 100 | Total: 14.0077 | Graph: 2.0168 | KLD: 1.0821 | Omics1: 4.5880 | Omics2: 6.3207
Epoch 110 | Total: 13.4016 | Graph: 1.7337 | KLD: 1.0374 | Omics1: 4.4916 | Omics2: 6.1

In [97]:
z_2 = train_vgae(vgae2, data = data_r2, epochs = 1000, device = device)

Epoch 010 | Total: 9.4346 | Graph: 0.8645 | KLD: 0.9716 | Omics1: 3.6076 | Omics2: 3.9909
Epoch 020 | Total: 9.3642 | Graph: 0.8649 | KLD: 1.0025 | Omics1: 3.5748 | Omics2: 3.9220
Epoch 030 | Total: 9.3527 | Graph: 0.8402 | KLD: 0.9775 | Omics1: 3.5779 | Omics2: 3.9571
Epoch 040 | Total: 9.3032 | Graph: 0.8179 | KLD: 0.9972 | Omics1: 3.5645 | Omics2: 3.9235
Epoch 050 | Total: 9.2979 | Graph: 0.8254 | KLD: 0.9745 | Omics1: 3.5754 | Omics2: 3.9225
Epoch 060 | Total: 9.2948 | Graph: 0.8024 | KLD: 0.9784 | Omics1: 3.5765 | Omics2: 3.9375
Epoch 070 | Total: 9.2810 | Graph: 0.8310 | KLD: 0.9835 | Omics1: 3.5669 | Omics2: 3.8997
Epoch 080 | Total: 9.3162 | Graph: 0.8194 | KLD: 0.9622 | Omics1: 3.5871 | Omics2: 3.9475
Epoch 090 | Total: 9.2231 | Graph: 0.8136 | KLD: 0.9903 | Omics1: 3.5509 | Omics2: 3.8683
Epoch 100 | Total: 9.3117 | Graph: 0.8655 | KLD: 0.9862 | Omics1: 3.5451 | Omics2: 3.9149
Epoch 110 | Total: 9.2852 | Graph: 0.8640 | KLD: 1.0067 | Omics1: 3.5300 | Omics2: 3.8845
Epoch 120 

In [124]:
# Convert the embedding to a DataFrame for clustering
z_np = z_2.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r2.copy()
num_cluster_r2 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r2,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )

In [141]:
# Clustering
cluster_start = time.time()
bayes.fit(df_z)
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)

cluster_end = time.time()
cluster_time = cluster_end - cluster_start
print(f"Clustering time: {cluster_time:.2f}s")

refine_start = time.time()
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 70)
refine_end = time.time()

refine_time = refine_end - refine_start
total_time = refine_end - total_start
print(f"Refinement time: {refine_time:.2f}s")
print(f"Total time: {total_time:.2f}s")

Clustering time: 5.55s
Refinement time: 0.07s
Total time: 7340.52s


In [142]:
dataset = "MISAR_seq_mouse_E15_brain_data"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing TAC_RNA_Seq_MouseBrainheader
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2580
NMI: 0.4368
HOM: 0.4014


### Data R3

In [20]:
total_start = time.time()
vgae_start = time.time()

vgae3 = VGAE(
    dropout = 0.2,
    in_omics1 = data_r3.x_omics1.shape[1],
    in_omics2 = data_r3.x_omics2.shape[1],
    recon_omics1_dim = data_r3.x_omics1.shape[1],
    recon_omics2_dim = data_r3.x_omics2.shape[1],
    recon_spatial_dim = data_r3.edge_index.shape[1]
    ).to(device)   

z = train_vgae(vgae3, data = data_r3, epochs = 1000, device = device)

vgae_end = time.time()
vgae_time = vgae_end - vgae_start
print(f"VGAE time: {vgae_time:.2f}s")

Epoch 010 | Total: 9.6949 | Graph: 1.7425 | KLD: 0.0409 | Omics1: 6.8727 | Omics2: 1.0388
Epoch 020 | Total: 8.9659 | Graph: 0.9608 | KLD: 0.3981 | Omics1: 6.6274 | Omics2: 0.9797
Epoch 030 | Total: 8.6121 | Graph: 0.8025 | KLD: 0.4451 | Omics1: 6.4236 | Omics2: 0.9409
Epoch 040 | Total: 8.5259 | Graph: 1.5422 | KLD: 0.4101 | Omics1: 5.7020 | Omics2: 0.8716
Epoch 050 | Total: 8.0565 | Graph: 1.6145 | KLD: 0.4796 | Omics1: 5.1161 | Omics2: 0.8463
Epoch 060 | Total: 7.8113 | Graph: 1.7557 | KLD: 0.4906 | Omics1: 4.7289 | Omics2: 0.8361
Epoch 070 | Total: 7.4372 | Graph: 1.6547 | KLD: 0.4790 | Omics1: 4.4867 | Omics2: 0.8169
Epoch 080 | Total: 7.1372 | Graph: 1.5884 | KLD: 0.4840 | Omics1: 4.2709 | Omics2: 0.7938
Epoch 090 | Total: 6.8951 | Graph: 1.4836 | KLD: 0.5221 | Omics1: 4.1117 | Omics2: 0.7776
Epoch 100 | Total: 6.6629 | Graph: 1.3672 | KLD: 0.5358 | Omics1: 3.9934 | Omics2: 0.7665
Epoch 110 | Total: 6.4844 | Graph: 1.2473 | KLD: 0.5364 | Omics1: 3.9401 | Omics2: 0.7605
Epoch 120 

In [55]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r3.copy()
num_cluster_r3 = len(adata.obs['true_label_level_0'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )

In [56]:
# Clustering
bayes.fit(df_z)
cluster_start = time.time()
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)

cluster_end = time.time()
cluster_time = cluster_end - cluster_start
print(f"Clustering time: {cluster_time:.2f}s")

refine_start = time.time()
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)
refine_end = time.time()

refine_time = refine_end - refine_start
total_time = refine_end - total_start
print(f"Refinement time: {refine_time:.2f}s")
print(f"Total time: {total_time:.2f}s")

Clustering time: 0.02s
Refinement time: 0.03s
Total time: 2210.91s


/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


In [57]:
dataset = "Human_Lymph_Node (Level 0)"

# True labels
labels_true = adata.obs['true_label_level_0'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2047
NMI: 0.3521
HOM: 0.4023


In [28]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r3.copy()
num_cluster_r3 = len(adata.obs['true_label_level_1'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )

# Clustering
cluster_start = time.time()

bayes.fit(df_z)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=8, random_state=42)

In [39]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)

cluster_end = time.time()
cluster_time = cluster_end - cluster_start
print(f"Clustering time: {cluster_time:.2f}s")

refine_start = time.time()
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 8)
refine_end = time.time()

refine_time = refine_end - refine_start
total_time = refine_end - total_start
print(f"Refinement time: {refine_time:.2f}s")
print(f"Total time: {total_time:.2f}s")

Clustering time: 275.41s
Refinement time: 0.04s
Total time: 1689.46s


In [40]:
dataset = "Human_Lymph_Node (Level 1)"

# True labels
labels_true = adata.obs['true_label_level_1'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "VGAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.6470
NMI: 0.4844
HOM: 0.4763


## GAE

### Data 3

In [16]:
gae3 = GAE(
    dropout=0.2,
    in_omics1=data_3.x_omics1.shape[1],
    in_omics2=data_3.x_omics2.shape[1],
    recon_omics1_dim=data_3.x_omics1.shape[1],
    recon_omics2_dim=data_3.x_omics2.shape[1],
    recon_spatial_dim=data_3.edge_index.shape[1],
)

z = train_gae(gae3, data_3, epochs=300, device=device)

Epoch 001 | Total: 12.1263 | Graph: 0.6822 | Omics1: 10.2759 | Omics2: 1.1683
Epoch 002 | Total: 12.0744 | Graph: 0.6794 | Omics1: 10.2311 | Omics2: 1.1639
Epoch 003 | Total: 12.0210 | Graph: 0.6747 | Omics1: 10.1865 | Omics2: 1.1598
Epoch 004 | Total: 11.9575 | Graph: 0.6680 | Omics1: 10.1345 | Omics2: 1.1550
Epoch 005 | Total: 11.8826 | Graph: 0.6595 | Omics1: 10.0737 | Omics2: 1.1494
Epoch 006 | Total: 11.7960 | Graph: 0.6500 | Omics1: 10.0031 | Omics2: 1.1429
Epoch 007 | Total: 11.6966 | Graph: 0.6408 | Omics1: 9.9207 | Omics2: 1.1351
Epoch 008 | Total: 11.5800 | Graph: 0.6338 | Omics1: 9.8202 | Omics2: 1.1260
Epoch 009 | Total: 11.4482 | Graph: 0.6328 | Omics1: 9.7002 | Omics2: 1.1152
Epoch 010 | Total: 11.2976 | Graph: 0.6305 | Omics1: 9.5632 | Omics2: 1.1039
Epoch 011 | Total: 11.1311 | Graph: 0.6429 | Omics1: 9.3977 | Omics2: 1.0905
Epoch 012 | Total: 10.9650 | Graph: 0.6758 | Omics1: 9.2130 | Omics2: 1.0762
Epoch 013 | Total: 10.8779 | Graph: 0.8286 | Omics1: 8.9874 | Omics2: 

In [18]:
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

adata = adata_rna_3.copy()
num_cluster_3 = len(adata.obs['label'].unique())
bayes = BayesianGaussianMixture(n_components = num_cluster_3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [19]:
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [ ]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3842
NMI: 0.6146
HOM: 0.6102


### Data 4

In [25]:
gae4 = GAE(
    dropout=0.2,
    in_omics1=data_4.x_omics1.shape[1],
    in_omics2=data_4.x_omics2.shape[1],
    recon_omics1_dim=data_4.x_omics1.shape[1],
    recon_omics2_dim=data_4.x_omics2.shape[1],
    recon_spatial_dim=data_4.edge_index.shape[1],
)

z = train_gae(gae4, data_4, epochs=500, device=device)

Epoch 001 | Total: 92.8036 | Graph: 0.9284 | Omics1: 90.7090 | Omics2: 1.1662
Epoch 002 | Total: 92.1516 | Graph: 0.7474 | Omics1: 90.2430 | Omics2: 1.1612
Epoch 003 | Total: 91.6383 | Graph: 0.6846 | Omics1: 89.7964 | Omics2: 1.1573
Epoch 004 | Total: 91.1119 | Graph: 0.6762 | Omics1: 89.2832 | Omics2: 1.1525
Epoch 005 | Total: 90.6017 | Graph: 0.7500 | Omics1: 88.7053 | Omics2: 1.1464
Epoch 006 | Total: 90.1480 | Graph: 0.9579 | Omics1: 88.0517 | Omics2: 1.1384
Epoch 007 | Total: 89.6599 | Graph: 1.3402 | Omics1: 87.1925 | Omics2: 1.1272
Epoch 008 | Total: 88.9481 | Graph: 1.6531 | Omics1: 86.1805 | Omics2: 1.1144
Epoch 009 | Total: 88.1471 | Graph: 2.0136 | Omics1: 85.0319 | Omics2: 1.1016
Epoch 010 | Total: 87.3731 | Graph: 2.7423 | Omics1: 83.5415 | Omics2: 1.0893
Epoch 011 | Total: 86.7091 | Graph: 3.7404 | Omics1: 81.8891 | Omics2: 1.0796
Epoch 012 | Total: 86.0003 | Graph: 5.0037 | Omics1: 79.9233 | Omics2: 1.0734
Epoch 013 | Total: 84.7563 | Graph: 6.1462 | Omics1: 77.5357 | O

In [ ]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_4.copy()
num_cluster_4 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_4,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

In [ ]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [ ]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3911
NMI: 0.6304
HOM: 0.6232


### Data 5

In [18]:
gae5 = GAE(
    dropout=0.2,
    in_omics1=data_5.x_omics1.shape[1],
    in_omics2=data_5.x_omics2.shape[1],
    recon_omics1_dim=data_5.x_omics1.shape[1],
    recon_omics2_dim=data_5.x_omics2.shape[1],
    recon_spatial_dim=data_5.edge_index.shape[1],
)

z = train_gae(gae5, data_5, epochs=500, device=device)

Epoch 001 | Total: 12.1237 | Graph: 0.6831 | Omics1: 10.2704 | Omics2: 1.1702
Epoch 002 | Total: 12.0585 | Graph: 0.6739 | Omics1: 10.2198 | Omics2: 1.1648
Epoch 003 | Total: 11.9909 | Graph: 0.6649 | Omics1: 10.1664 | Omics2: 1.1596
Epoch 004 | Total: 11.9136 | Graph: 0.6542 | Omics1: 10.1048 | Omics2: 1.1546
Epoch 005 | Total: 11.8246 | Graph: 0.6425 | Omics1: 10.0331 | Omics2: 1.1490
Epoch 006 | Total: 11.7212 | Graph: 0.6326 | Omics1: 9.9456 | Omics2: 1.1429
Epoch 007 | Total: 11.6040 | Graph: 0.6249 | Omics1: 9.8429 | Omics2: 1.1362
Epoch 008 | Total: 11.4754 | Graph: 0.6240 | Omics1: 9.7233 | Omics2: 1.1281
Epoch 009 | Total: 11.3339 | Graph: 0.6334 | Omics1: 9.5823 | Omics2: 1.1182
Epoch 010 | Total: 11.1856 | Graph: 0.6551 | Omics1: 9.4233 | Omics2: 1.1072
Epoch 011 | Total: 11.0246 | Graph: 0.6906 | Omics1: 9.2404 | Omics2: 1.0936
Epoch 012 | Total: 10.9567 | Graph: 0.8454 | Omics1: 9.0324 | Omics2: 1.0789
Epoch 013 | Total: 11.0473 | Graph: 1.1800 | Omics1: 8.8048 | Omics2: 1

In [19]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_5.copy()
num_cluster_5 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_5,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [ ]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [21]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3942
NMI: 0.6276
HOM: 0.6152


### Data 6

In [21]:
gae6 = GAE(
    dropout=0.2,
    in_omics1=data_6.x_omics1.shape[1],
    in_omics2=data_6.x_omics2.shape[1],
    recon_omics1_dim=data_6.x_omics1.shape[1],
    recon_omics2_dim=data_6.x_omics2.shape[1],
    recon_spatial_dim=data_6.edge_index.shape[1],
)

z = train_gae(gae6, data_6, epochs=500, device=device)

Epoch 001 | Total: 93.1096 | Graph: 1.0000 | Omics1: 90.9405 | Omics2: 1.1691
Epoch 002 | Total: 92.6211 | Graph: 0.7588 | Omics1: 90.7019 | Omics2: 1.1604
Epoch 003 | Total: 92.2702 | Graph: 0.6719 | Omics1: 90.4440 | Omics2: 1.1543
Epoch 004 | Total: 91.9376 | Graph: 0.6382 | Omics1: 90.1507 | Omics2: 1.1487
Epoch 005 | Total: 91.5626 | Graph: 0.6301 | Omics1: 89.7896 | Omics2: 1.1429
Epoch 006 | Total: 91.1596 | Graph: 0.6444 | Omics1: 89.3800 | Omics2: 1.1352
Epoch 007 | Total: 90.7108 | Graph: 0.6949 | Omics1: 88.8892 | Omics2: 1.1267
Epoch 008 | Total: 90.1956 | Graph: 0.7794 | Omics1: 88.3014 | Omics2: 1.1148
Epoch 009 | Total: 89.6071 | Graph: 0.9067 | Omics1: 87.5995 | Omics2: 1.1009
Epoch 010 | Total: 89.0289 | Graph: 1.1911 | Omics1: 86.7530 | Omics2: 1.0847
Epoch 011 | Total: 88.7782 | Graph: 1.9313 | Omics1: 85.7800 | Omics2: 1.0669
Epoch 012 | Total: 88.7437 | Graph: 3.1538 | Omics1: 84.5418 | Omics2: 1.0481
Epoch 013 | Total: 88.6593 | Graph: 4.4805 | Omics1: 83.1462 | O

In [22]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_6.copy()
num_cluster_6 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_6,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [23]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [24]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4261
NMI: 0.6418
HOM: 0.6375


### Data R1

In [13]:
gaer1 = GAE(
    dropout=0.2,
    in_omics1=data_r1.x_omics1.shape[1],
    in_omics2=data_r1.x_omics2.shape[1],
    recon_omics1_dim=data_r1.x_omics1.shape[1],
    recon_omics2_dim=data_r1.x_omics2.shape[1],
    recon_spatial_dim=data_r1.edge_index.shape[1],
)

z = train_gae(gaer1, data_r1, epochs=200, device=device)

Epoch 001 | Total: 6.1831 | Graph: 0.6873 | Omics1: 4.4747 | Omics2: 1.0211
Epoch 002 | Total: 6.1623 | Graph: 0.6731 | Omics1: 4.4702 | Omics2: 1.0190
Epoch 003 | Total: 6.1491 | Graph: 0.6661 | Omics1: 4.4654 | Omics2: 1.0175
Epoch 004 | Total: 6.1357 | Graph: 0.6594 | Omics1: 4.4601 | Omics2: 1.0162
Epoch 005 | Total: 6.1209 | Graph: 0.6525 | Omics1: 4.4535 | Omics2: 1.0150
Epoch 006 | Total: 6.1052 | Graph: 0.6452 | Omics1: 4.4462 | Omics2: 1.0138
Epoch 007 | Total: 6.0865 | Graph: 0.6365 | Omics1: 4.4374 | Omics2: 1.0126
Epoch 008 | Total: 6.0669 | Graph: 0.6289 | Omics1: 4.4269 | Omics2: 1.0111
Epoch 009 | Total: 6.0472 | Graph: 0.6209 | Omics1: 4.4166 | Omics2: 1.0098
Epoch 010 | Total: 6.0421 | Graph: 0.6307 | Omics1: 4.4032 | Omics2: 1.0081
Epoch 011 | Total: 6.0438 | Graph: 0.6485 | Omics1: 4.3889 | Omics2: 1.0064
Epoch 012 | Total: 6.0368 | Graph: 0.6579 | Omics1: 4.3742 | Omics2: 1.0046
Epoch 013 | Total: 6.0509 | Graph: 0.6900 | Omics1: 4.3582 | Omics2: 1.0028
Epoch 014 | 

In [14]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r1.copy()
num_cluster_r1 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r1,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=9, random_state=42)

In [15]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [16]:
dataset = "ATAC_RNA_Seq_MouseBrain"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3549
NMI: 0.5387
HOM: 0.5106


### Data R2

In [77]:
gaer2 = GAE(
    dropout=0.2,
    in_omics1=data_r2.x_omics1.shape[1],
    in_omics2=data_r2.x_omics2.shape[1],
    recon_omics1_dim=data_r2.x_omics1.shape[1],
    recon_omics2_dim=data_r2.x_omics2.shape[1],
    recon_spatial_dim=data_r2.edge_index.shape[1],
)

z = train_gae(gaer2, data_r2, epochs=500, device=device)

Epoch 001 | Total: 20.1187 | Graph: 0.7280 | Omics1: 6.7276 | Omics2: 12.6631


Epoch 002 | Total: 20.0210 | Graph: 0.6454 | Omics1: 6.7222 | Omics2: 12.6534
Epoch 003 | Total: 19.9786 | Graph: 0.6209 | Omics1: 6.7155 | Omics2: 12.6422
Epoch 004 | Total: 19.9488 | Graph: 0.6110 | Omics1: 6.7067 | Omics2: 12.6311
Epoch 005 | Total: 19.9144 | Graph: 0.6038 | Omics1: 6.6970 | Omics2: 12.6135
Epoch 006 | Total: 19.8776 | Graph: 0.6001 | Omics1: 6.6852 | Omics2: 12.5923
Epoch 007 | Total: 19.8439 | Graph: 0.6086 | Omics1: 6.6699 | Omics2: 12.5655
Epoch 008 | Total: 19.8000 | Graph: 0.6097 | Omics1: 6.6528 | Omics2: 12.5374
Epoch 009 | Total: 19.7767 | Graph: 0.6524 | Omics1: 6.6310 | Omics2: 12.4934
Epoch 010 | Total: 19.7468 | Graph: 0.6855 | Omics1: 6.6083 | Omics2: 12.4530
Epoch 011 | Total: 19.7379 | Graph: 0.7430 | Omics1: 6.5859 | Omics2: 12.4090
Epoch 012 | Total: 19.7590 | Graph: 0.8404 | Omics1: 6.5594 | Omics2: 12.3592
Epoch 013 | Total: 19.7603 | Graph: 0.9424 | Omics1: 6.5293 | Omics2: 12.2885
Epoch 014 | Total: 19.7050 | Graph: 0.9888 | Omics1: 6.5010 | Om

In [78]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r2.copy()
num_cluster_r2 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r2,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=7, random_state=42)

In [79]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [80]:
dataset = "MISAR_seq_mouse_E15_brain_data"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GAE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: -0.0184
NMI: 0.2387
HOM: 0.1954


## GATE

### Data 3

In [18]:
epochs = 300

gate3 = GATE(
    dropout = 0.2,
    in_omics1 = data_3.x_omics1.shape[1],
    in_omics2 = data_3.x_omics2.shape[1],
).to(device)
z = train_gate(gate3, data = data_3, epochs = epochs, device = device)

Epoch 001 | Total: 11.4531 | Omics1: 10.2847 | Omics2: 1.1682 | Smooth: 0.0011 | Reg: 0.0069
Epoch 002 | Total: 11.3966 | Omics1: 10.2330 | Omics2: 1.1634 | Smooth: 0.0011 | Reg: 0.0075
Epoch 003 | Total: 11.3368 | Omics1: 10.1779 | Omics2: 1.1588 | Smooth: 0.0013 | Reg: 0.0091
Epoch 004 | Total: 11.2684 | Omics1: 10.1143 | Omics2: 1.1540 | Smooth: 0.0015 | Reg: 0.0120
Epoch 005 | Total: 11.1905 | Omics1: 10.0422 | Omics2: 1.1481 | Smooth: 0.0019 | Reg: 0.0160
Epoch 006 | Total: 11.0976 | Omics1: 9.9569 | Omics2: 1.1404 | Smooth: 0.0025 | Reg: 0.0223
Epoch 007 | Total: 10.9855 | Omics1: 9.8543 | Omics2: 1.1308 | Smooth: 0.0032 | Reg: 0.0312
Epoch 008 | Total: 10.8510 | Omics1: 9.7310 | Omics2: 1.1195 | Smooth: 0.0043 | Reg: 0.0433
Epoch 009 | Total: 10.6845 | Omics1: 9.5780 | Omics2: 1.1059 | Smooth: 0.0057 | Reg: 0.0610
Epoch 010 | Total: 10.4819 | Omics1: 9.3915 | Omics2: 1.0896 | Smooth: 0.0074 | Reg: 0.0851
Epoch 011 | Total: 10.2438 | Omics1: 9.1718 | Omics2: 1.0709 | Smooth: 0.01

In [19]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_3.copy()
num_cluster_3 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [20]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [21]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4537
NMI: 0.6749
HOM: 0.6769


### Data 4

In [27]:
epochs = 300

gate4 = GATE(
    dropout = 0.2,
    in_omics1 = data_4.x_omics1.shape[1],
    in_omics2 = data_4.x_omics2.shape[1],
).to(device)
z = train_gate(gate4, data = data_4, epochs = epochs, device = device)

Epoch 001 | Total: 92.1203 | Omics1: 90.9470 | Omics2: 1.1727 | Smooth: 0.0062 | Reg: 0.0190
Epoch 002 | Total: 91.7576 | Omics1: 90.5907 | Omics2: 1.1662 | Smooth: 0.0062 | Reg: 0.0193
Epoch 003 | Total: 91.4535 | Omics1: 90.2931 | Omics2: 1.1598 | Smooth: 0.0068 | Reg: 0.0242
Epoch 004 | Total: 91.1334 | Omics1: 89.9793 | Omics2: 1.1533 | Smooth: 0.0076 | Reg: 0.0325
Epoch 005 | Total: 90.7626 | Omics1: 89.6170 | Omics2: 1.1446 | Smooth: 0.0092 | Reg: 0.0456
Epoch 006 | Total: 90.2999 | Omics1: 89.1651 | Omics2: 1.1336 | Smooth: 0.0114 | Reg: 0.0667
Epoch 007 | Total: 89.7525 | Omics1: 88.6288 | Omics2: 1.1222 | Smooth: 0.0144 | Reg: 0.0965
Epoch 008 | Total: 89.0587 | Omics1: 87.9480 | Omics2: 1.1087 | Smooth: 0.0185 | Reg: 0.1409
Epoch 009 | Total: 88.2294 | Omics1: 87.1307 | Omics2: 1.0960 | Smooth: 0.0252 | Reg: 0.2033
Epoch 010 | Total: 87.1881 | Omics1: 86.0981 | Omics2: 1.0864 | Smooth: 0.0329 | Reg: 0.2942
Epoch 011 | Total: 85.8802 | Omics1: 84.7929 | Omics2: 1.0824 | Smooth

In [42]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_4.copy()
num_cluster_4 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_4,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [43]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [44]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4163
NMI: 0.6510
HOM: 0.6515


### Data 5

In [45]:
epochs = 300

gate5 = GATE(
    dropout = 0.2,
    in_omics1 = data_5.x_omics1.shape[1],
    in_omics2 = data_5.x_omics2.shape[1],
).to(device)
z = train_gate(gate5, data = data_5, epochs = epochs, device = device)

Epoch 001 | Total: 11.4486 | Omics1: 10.2793 | Omics2: 1.1691 | Smooth: 0.0009 | Reg: 0.0054
Epoch 002 | Total: 11.3950 | Omics1: 10.2298 | Omics2: 1.1651 | Smooth: 0.0010 | Reg: 0.0058
Epoch 003 | Total: 11.3423 | Omics1: 10.1816 | Omics2: 1.1606 | Smooth: 0.0011 | Reg: 0.0072
Epoch 004 | Total: 11.2867 | Omics1: 10.1315 | Omics2: 1.1551 | Smooth: 0.0013 | Reg: 0.0096
Epoch 005 | Total: 11.2180 | Omics1: 10.0699 | Omics2: 1.1480 | Smooth: 0.0016 | Reg: 0.0134
Epoch 006 | Total: 11.1344 | Omics1: 9.9947 | Omics2: 1.1395 | Smooth: 0.0021 | Reg: 0.0189
Epoch 007 | Total: 11.0281 | Omics1: 9.8992 | Omics2: 1.1286 | Smooth: 0.0027 | Reg: 0.0271
Epoch 008 | Total: 10.8973 | Omics1: 9.7816 | Omics2: 1.1153 | Smooth: 0.0036 | Reg: 0.0393
Epoch 009 | Total: 10.7408 | Omics1: 9.6399 | Omics2: 1.1003 | Smooth: 0.0048 | Reg: 0.0565
Epoch 010 | Total: 10.5521 | Omics1: 9.4686 | Omics2: 1.0828 | Smooth: 0.0065 | Reg: 0.0811
Epoch 011 | Total: 10.3291 | Omics1: 9.2653 | Omics2: 1.0628 | Smooth: 0.00

In [46]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_5.copy()
num_cluster_5 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_5,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

c:\Users\hana2\miniconda3\envs\SpatialGlue\lib\site-packages\sklearn\mixture\_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=15, random_state=42)

In [47]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [48]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4592
NMI: 0.6755
HOM: 0.6747


### Data 6

In [49]:
epochs = 300

gate6 = GATE(
    dropout = 0.2,
    in_omics1 = data_6.x_omics1.shape[1],
    in_omics2 = data_6.x_omics2.shape[1],
).to(device)
z = train_gate(gate6, data = data_6, epochs = epochs, device = device)

Epoch 001 | Total: 92.0996 | Omics1: 90.9302 | Omics2: 1.1687 | Smooth: 0.0066 | Reg: 0.0232
Epoch 002 | Total: 91.7210 | Omics1: 90.5569 | Omics2: 1.1635 | Smooth: 0.0066 | Reg: 0.0251
Epoch 003 | Total: 91.3189 | Omics1: 90.1598 | Omics2: 1.1583 | Smooth: 0.0077 | Reg: 0.0346
Epoch 004 | Total: 90.8758 | Omics1: 89.7236 | Omics2: 1.1513 | Smooth: 0.0093 | Reg: 0.0505
Epoch 005 | Total: 90.2783 | Omics1: 89.1344 | Omics2: 1.1426 | Smooth: 0.0119 | Reg: 0.0778
Epoch 006 | Total: 89.5496 | Omics1: 88.4171 | Omics2: 1.1309 | Smooth: 0.0157 | Reg: 0.1175
Epoch 007 | Total: 88.5978 | Omics1: 87.4792 | Omics2: 1.1164 | Smooth: 0.0208 | Reg: 0.1792
Epoch 008 | Total: 87.4388 | Omics1: 86.3361 | Omics2: 1.0995 | Smooth: 0.0289 | Reg: 0.2685
Epoch 009 | Total: 85.9723 | Omics1: 84.8850 | Omics2: 1.0829 | Smooth: 0.0396 | Reg: 0.4037
Epoch 010 | Total: 84.2328 | Omics1: 83.1586 | Omics2: 1.0682 | Smooth: 0.0544 | Reg: 0.5869
Epoch 011 | Total: 82.0835 | Omics1: 81.0163 | Omics2: 1.0591 | Smooth

In [50]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_6.copy()
num_cluster_6 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_6,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [51]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [52]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4112
NMI: 0.6410
HOM: 0.6394


### Data R1

In [18]:
epochs = 400

gater1 = GATE(
    dropout = 0.2,
    in_omics1 = data_r1.x_omics1.shape[1],
    in_omics2 = data_r1.x_omics2.shape[1],
).to(device)
z = train_gate(gater1, data = data_r1, epochs = epochs, device = device)

Epoch 001 | Total: 5.4926 | Omics1: 4.4765 | Omics2: 1.0159 | Smooth: 0.0022 | Reg: 0.0078
Epoch 002 | Total: 5.4855 | Omics1: 4.4707 | Omics2: 1.0146 | Smooth: 0.0022 | Reg: 0.0077
Epoch 003 | Total: 5.4786 | Omics1: 4.4648 | Omics2: 1.0136 | Smooth: 0.0022 | Reg: 0.0083
Epoch 004 | Total: 5.4707 | Omics1: 4.4579 | Omics2: 1.0126 | Smooth: 0.0024 | Reg: 0.0099
Epoch 005 | Total: 5.4615 | Omics1: 4.4497 | Omics2: 1.0115 | Smooth: 0.0027 | Reg: 0.0127
Epoch 006 | Total: 5.4513 | Omics1: 4.4405 | Omics2: 1.0105 | Smooth: 0.0032 | Reg: 0.0167
Epoch 007 | Total: 5.4389 | Omics1: 4.4292 | Omics2: 1.0093 | Smooth: 0.0037 | Reg: 0.0228
Epoch 008 | Total: 5.4229 | Omics1: 4.4143 | Omics2: 1.0081 | Smooth: 0.0047 | Reg: 0.0324
Epoch 009 | Total: 5.4048 | Omics1: 4.3973 | Omics2: 1.0068 | Smooth: 0.0057 | Reg: 0.0457
Epoch 010 | Total: 5.3838 | Omics1: 4.3774 | Omics2: 1.0056 | Smooth: 0.0074 | Reg: 0.0639
Epoch 011 | Total: 5.3578 | Omics1: 4.3525 | Omics2: 1.0042 | Smooth: 0.0100 | Reg: 0.0918

In [19]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r1.copy()
num_cluster_r1 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r1,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=9, random_state=42)

In [20]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [21]:
dataset = "ATAC_RNA_Seq_MouseBrain"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2527
NMI: 0.5126
HOM: 0.4835


### Data R2

In [81]:
epochs = 300

gater2 = GATE(
    dropout = 0.2,
    in_omics1 = data_r2.x_omics1.shape[1],
    in_omics2 = data_r2.x_omics2.shape[1],
).to(device)
z = train_gate(gater2, data = data_r2, epochs = epochs, device = device)

Epoch 001 | Total: 19.4143 | Omics1: 6.7375 | Omics2: 12.6753 | Smooth: 0.0144 | Reg: 0.0360
Epoch 002 | Total: 19.3808 | Omics1: 6.7243 | Omics2: 12.6550 | Smooth: 0.0152 | Reg: 0.0383
Epoch 003 | Total: 19.3520 | Omics1: 6.7099 | Omics2: 12.6406 | Smooth: 0.0146 | Reg: 0.0435
Epoch 004 | Total: 19.3082 | Omics1: 6.6897 | Omics2: 12.6167 | Smooth: 0.0181 | Reg: 0.0622
Epoch 005 | Total: 19.2591 | Omics1: 6.6690 | Omics2: 12.5880 | Smooth: 0.0197 | Reg: 0.0856
Epoch 006 | Total: 19.1897 | Omics1: 6.6391 | Omics2: 12.5479 | Smooth: 0.0262 | Reg: 0.1327
Epoch 007 | Total: 19.1069 | Omics1: 6.6019 | Omics2: 12.5013 | Smooth: 0.0352 | Reg: 0.1980
Epoch 008 | Total: 19.0130 | Omics1: 6.5601 | Omics2: 12.4481 | Smooth: 0.0455 | Reg: 0.2902
Epoch 009 | Total: 18.8852 | Omics1: 6.5081 | Omics2: 12.3707 | Smooth: 0.0599 | Reg: 0.4242
Epoch 010 | Total: 18.7269 | Omics1: 6.4398 | Omics2: 12.2779 | Smooth: 0.0854 | Reg: 0.6553
Epoch 011 | Total: 18.5264 | Omics1: 6.3611 | Omics2: 12.1520 | Smooth

In [82]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r2.copy()
num_cluster_r2 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r2,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=7, random_state=42)

In [83]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [84]:
dataset = "MISAR_seq_mouse_E15_brain_data"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "GATE"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2656
NMI: 0.4415
HOM: 0.4389


## BGRL

### Data 3

In [17]:
bgrl3 = BGRL(
    in_omics1=data_3.x_omics1.shape[1],
    in_omics2=data_3.x_omics2.shape[1]
)

bgrl3, loss_hist3 = train_bgrl(bgrl3, data_3, epochs=300)

z = get_embedding_bgrl(bgrl3, data_3)

Epoch 001 | Loss: 3.4607
Epoch 011 | Loss: 1.9048
Epoch 021 | Loss: 1.0965
Epoch 031 | Loss: 0.7435
Epoch 041 | Loss: 0.5884
Epoch 051 | Loss: 0.4855
Epoch 061 | Loss: 0.4101
Epoch 071 | Loss: 0.3496
Epoch 081 | Loss: 0.2998
Epoch 091 | Loss: 0.2662
Epoch 101 | Loss: 0.2375
Epoch 111 | Loss: 0.2136
Epoch 121 | Loss: 0.1983
Epoch 131 | Loss: 0.1843
Epoch 141 | Loss: 0.1729
Epoch 151 | Loss: 0.1667
Epoch 161 | Loss: 0.1545
Epoch 171 | Loss: 0.1459
Epoch 181 | Loss: 0.1351
Epoch 191 | Loss: 0.1255
Epoch 201 | Loss: 0.1198
Epoch 211 | Loss: 0.1151
Epoch 221 | Loss: 0.1073
Epoch 231 | Loss: 0.1065
Epoch 241 | Loss: 0.0998
Epoch 251 | Loss: 0.0961
Epoch 261 | Loss: 0.0911
Epoch 271 | Loss: 0.0920
Epoch 281 | Loss: 0.0881
Epoch 291 | Loss: 0.0863
Epoch 300 | Loss: 0.0828


In [18]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_3.copy()
num_cluster_3 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [19]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [ ]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3654
NMI: 0.5920
HOM: 0.5917


### Data 4

In [23]:
bgrl4 = BGRL(
    in_omics1=data_4.x_omics1.shape[1],
    in_omics2=data_4.x_omics2.shape[1]
)

bgrl4, loss_hist4 = train_bgrl(bgrl4, data_4, epochs=300)

z = get_embedding_bgrl(bgrl4, data_4)

Epoch 001 | Loss: 3.3313
Epoch 011 | Loss: 1.4490
Epoch 021 | Loss: 0.9418
Epoch 031 | Loss: 0.7172
Epoch 041 | Loss: 0.5972
Epoch 051 | Loss: 0.5002
Epoch 061 | Loss: 0.4290
Epoch 071 | Loss: 0.3649
Epoch 081 | Loss: 0.3107
Epoch 091 | Loss: 0.2710
Epoch 101 | Loss: 0.2389
Epoch 111 | Loss: 0.2238
Epoch 121 | Loss: 0.2028
Epoch 131 | Loss: 0.1909
Epoch 141 | Loss: 0.1818
Epoch 151 | Loss: 0.1677
Epoch 161 | Loss: 0.1632
Epoch 171 | Loss: 0.1512
Epoch 181 | Loss: 0.1474
Epoch 191 | Loss: 0.1413
Epoch 201 | Loss: 0.1357
Epoch 211 | Loss: 0.1309
Epoch 221 | Loss: 0.1261
Epoch 231 | Loss: 0.1249
Epoch 241 | Loss: 0.1190
Epoch 251 | Loss: 0.1171
Epoch 261 | Loss: 0.1146
Epoch 271 | Loss: 0.1160
Epoch 281 | Loss: 0.1073
Epoch 291 | Loss: 0.1069
Epoch 300 | Loss: 0.1071


In [24]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_4.copy()
num_cluster_4 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_4,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [25]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [26]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3752
NMI: 0.5988
HOM: 0.5981


### Data 5

In [27]:
bgrl5 = BGRL(
    in_omics1=data_5.x_omics1.shape[1],
    in_omics2=data_5.x_omics2.shape[1]
)

bgrl5, loss_hist5 = train_bgrl(bgrl5, data_5, epochs=300)

z = get_embedding_bgrl(bgrl5, data_5)

Epoch 001 | Loss: 3.6036
Epoch 011 | Loss: 1.7290
Epoch 021 | Loss: 0.9533
Epoch 031 | Loss: 0.5840
Epoch 041 | Loss: 0.4420
Epoch 051 | Loss: 0.3604
Epoch 061 | Loss: 0.3150
Epoch 071 | Loss: 0.2971
Epoch 081 | Loss: 0.2863
Epoch 091 | Loss: 0.2750
Epoch 101 | Loss: 0.2636
Epoch 111 | Loss: 0.2509
Epoch 121 | Loss: 0.2343
Epoch 131 | Loss: 0.2173
Epoch 141 | Loss: 0.2026
Epoch 151 | Loss: 0.1890
Epoch 161 | Loss: 0.1788
Epoch 171 | Loss: 0.1657
Epoch 181 | Loss: 0.1526
Epoch 191 | Loss: 0.1384
Epoch 201 | Loss: 0.1296
Epoch 211 | Loss: 0.1204
Epoch 221 | Loss: 0.1131
Epoch 231 | Loss: 0.1073
Epoch 241 | Loss: 0.1035
Epoch 251 | Loss: 0.0985
Epoch 261 | Loss: 0.0945
Epoch 271 | Loss: 0.0898
Epoch 281 | Loss: 0.0858
Epoch 291 | Loss: 0.0834
Epoch 300 | Loss: 0.0818


In [28]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_5.copy()
num_cluster_5 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_5,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [29]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [30]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4035
NMI: 0.6344
HOM: 0.6339


### Data 6

In [31]:
bgrl6 = BGRL(
    in_omics1=data_6.x_omics1.shape[1],
    in_omics2=data_6.x_omics2.shape[1]
)

bgrl6, loss_hist6 = train_bgrl(bgrl6, data_6, epochs=300)

z = get_embedding_bgrl(bgrl6, data_6)

Epoch 001 | Loss: 4.0857
Epoch 011 | Loss: 1.6777
Epoch 021 | Loss: 0.9554
Epoch 031 | Loss: 0.6923
Epoch 041 | Loss: 0.5512
Epoch 051 | Loss: 0.4598
Epoch 061 | Loss: 0.3919
Epoch 071 | Loss: 0.3429
Epoch 081 | Loss: 0.3014
Epoch 091 | Loss: 0.2731
Epoch 101 | Loss: 0.2452
Epoch 111 | Loss: 0.2248
Epoch 121 | Loss: 0.2063
Epoch 131 | Loss: 0.1921
Epoch 141 | Loss: 0.1817
Epoch 151 | Loss: 0.1707
Epoch 161 | Loss: 0.1631
Epoch 171 | Loss: 0.1558
Epoch 181 | Loss: 0.1467
Epoch 191 | Loss: 0.1411
Epoch 201 | Loss: 0.1353
Epoch 211 | Loss: 0.1281
Epoch 221 | Loss: 0.1241
Epoch 231 | Loss: 0.1175
Epoch 241 | Loss: 0.1138
Epoch 251 | Loss: 0.1105
Epoch 261 | Loss: 0.1071
Epoch 271 | Loss: 0.1050
Epoch 281 | Loss: 0.1021
Epoch 291 | Loss: 0.0992
Epoch 300 | Loss: 0.0956


In [32]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_6.copy()
num_cluster_6 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_6,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [33]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [ ]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3197
NMI: 0.5747
HOM: 0.5580


### Data R1

In [22]:
bgrlr1 = BGRL(
    in_omics1=data_r1.x_omics1.shape[1],
    in_omics2=data_r1.x_omics2.shape[1]
)

bgrlr1, loss_histr1 = train_bgrl(bgrlr1, data_r1, epochs=300)

z = get_embedding_bgrl(bgrlr1, data_r1)

Epoch 001 | Loss: 4.1275
Epoch 011 | Loss: 1.8406
Epoch 021 | Loss: 0.9523
Epoch 031 | Loss: 0.5711
Epoch 041 | Loss: 0.4135
Epoch 051 | Loss: 0.3260
Epoch 061 | Loss: 0.2633
Epoch 071 | Loss: 0.2212
Epoch 081 | Loss: 0.1944
Epoch 091 | Loss: 0.1773
Epoch 101 | Loss: 0.1668
Epoch 111 | Loss: 0.1572
Epoch 121 | Loss: 0.1491
Epoch 131 | Loss: 0.1409
Epoch 141 | Loss: 0.1302
Epoch 151 | Loss: 0.1230
Epoch 161 | Loss: 0.1170
Epoch 171 | Loss: 0.1127
Epoch 181 | Loss: 0.1079
Epoch 191 | Loss: 0.1033
Epoch 201 | Loss: 0.0994
Epoch 211 | Loss: 0.0931
Epoch 221 | Loss: 0.0891
Epoch 231 | Loss: 0.0838
Epoch 241 | Loss: 0.0820
Epoch 251 | Loss: 0.0780
Epoch 261 | Loss: 0.0770
Epoch 271 | Loss: 0.0754
Epoch 281 | Loss: 0.0725
Epoch 291 | Loss: 0.0711
Epoch 300 | Loss: 0.0703


In [23]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r1.copy()
num_cluster_r1 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r1,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=9, random_state=42)

In [24]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [25]:
dataset = "ATAC_RNA_Seq_MouseBrain"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3592
NMI: 0.5128
HOM: 0.5133


### Data R2

In [85]:
bgrlr2 = BGRL(
    in_omics1=data_r2.x_omics1.shape[1],
    in_omics2=data_r2.x_omics2.shape[1]
)

bgrlr2, loss_histr2 = train_bgrl(bgrlr2, data_r2, epochs=300)

z = get_embedding_bgrl(bgrlr2, data_r2)

Epoch 001 | Loss: 4.2836
Epoch 011 | Loss: 2.3837
Epoch 021 | Loss: 1.3590
Epoch 031 | Loss: 0.9534
Epoch 041 | Loss: 0.7197
Epoch 051 | Loss: 0.5677
Epoch 061 | Loss: 0.4630
Epoch 071 | Loss: 0.3865
Epoch 081 | Loss: 0.3316
Epoch 091 | Loss: 0.2857
Epoch 101 | Loss: 0.2509
Epoch 111 | Loss: 0.2222
Epoch 121 | Loss: 0.2004
Epoch 131 | Loss: 0.1802
Epoch 141 | Loss: 0.1653
Epoch 151 | Loss: 0.1510
Epoch 161 | Loss: 0.1411
Epoch 171 | Loss: 0.1302
Epoch 181 | Loss: 0.1270
Epoch 191 | Loss: 0.1179
Epoch 201 | Loss: 0.1133
Epoch 211 | Loss: 0.1083
Epoch 221 | Loss: 0.1056
Epoch 231 | Loss: 0.1029
Epoch 241 | Loss: 0.1016
Epoch 251 | Loss: 0.1001
Epoch 261 | Loss: 0.0975
Epoch 271 | Loss: 0.0956
Epoch 281 | Loss: 0.0937
Epoch 291 | Loss: 0.0921
Epoch 300 | Loss: 0.0903


In [86]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r2.copy()
num_cluster_r2 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r2,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=7, random_state=42)

In [87]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [88]:
dataset = "MISAR_seq_mouse_E15_brain_data"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "BGRL"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2528
NMI: 0.4253
HOM: 0.4364


## DGI

### Data 3

In [17]:
dgi3 = DGI(
    in_omics1=data_3.x_omics1.shape[1],
    in_omics2=data_3.x_omics2.shape[1]
)

dgi3, loss_hist3 = train_dgi(dgi3, data_3, epochs=300)

z = get_embedding_dgi(dgi3, data_3)

Epoch 001 | Loss: 1.4871
Epoch 011 | Loss: 0.7026
Epoch 021 | Loss: 0.2868
Epoch 031 | Loss: 0.2031
Epoch 041 | Loss: 0.1546
Epoch 051 | Loss: 0.1349
Epoch 061 | Loss: 0.1322
Epoch 071 | Loss: 0.1178
Epoch 081 | Loss: 0.1076
Epoch 091 | Loss: 0.0985
Epoch 101 | Loss: 0.0949
Epoch 111 | Loss: 0.0881
Epoch 121 | Loss: 0.0812
Epoch 131 | Loss: 0.0670
Epoch 141 | Loss: 0.0719
Epoch 151 | Loss: 0.0650
Epoch 161 | Loss: 0.0655
Epoch 171 | Loss: 0.0588
Epoch 181 | Loss: 0.0668
Epoch 191 | Loss: 0.0481
Epoch 201 | Loss: 0.0531
Epoch 211 | Loss: 0.0489
Epoch 221 | Loss: 0.0398
Epoch 231 | Loss: 0.0533
Epoch 241 | Loss: 0.0332
Epoch 251 | Loss: 0.0387
Epoch 261 | Loss: 0.0339
Epoch 271 | Loss: 0.0292
Epoch 281 | Loss: 0.0311
Epoch 291 | Loss: 0.0301
Epoch 300 | Loss: 0.0352


In [18]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_3.copy()
num_cluster_3 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_3,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [19]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [20]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3540
NMI: 0.5904
HOM: 0.5865


### Data 4

In [21]:
dgi4 = DGI(
    in_omics1=data_4.x_omics1.shape[1],
    in_omics2=data_4.x_omics2.shape[1]
)

dgi4, loss_hist4 = train_dgi(dgi4, data_4, epochs=300)

z = get_embedding_dgi(dgi4, data_4)

Epoch 001 | Loss: 2.5153
Epoch 011 | Loss: 1.1241
Epoch 021 | Loss: 0.7163
Epoch 031 | Loss: 0.4317
Epoch 041 | Loss: 0.3089
Epoch 051 | Loss: 0.2293
Epoch 061 | Loss: 0.1877
Epoch 071 | Loss: 0.1815
Epoch 081 | Loss: 0.1457
Epoch 091 | Loss: 0.1384
Epoch 101 | Loss: 0.1318
Epoch 111 | Loss: 0.1141
Epoch 121 | Loss: 0.1116
Epoch 131 | Loss: 0.1124
Epoch 141 | Loss: 0.0915
Epoch 151 | Loss: 0.0874
Epoch 161 | Loss: 0.0755
Epoch 171 | Loss: 0.0708
Epoch 181 | Loss: 0.0703
Epoch 191 | Loss: 0.0724
Epoch 201 | Loss: 0.0877
Epoch 211 | Loss: 0.0710
Epoch 221 | Loss: 0.0641
Epoch 231 | Loss: 0.0681
Epoch 241 | Loss: 0.0693
Epoch 251 | Loss: 0.0577
Epoch 261 | Loss: 0.0577
Epoch 271 | Loss: 0.0521
Epoch 281 | Loss: 0.0488
Epoch 291 | Loss: 0.0526
Epoch 300 | Loss: 0.0507


In [22]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_4.copy()
num_cluster_4 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_4,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [23]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [ ]:
dataset = "Amplify_CBMCs_nCells_5024_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3201
NMI: 0.5678
HOM: 0.5656


### Data 5

In [25]:
dgi5 = DGI(
    in_omics1=data_5.x_omics1.shape[1],
    in_omics2=data_5.x_omics2.shape[1]
)

dgi5, loss_hist5 = train_dgi(dgi5, data_5, epochs=300)

z = get_embedding_dgi(dgi5, data_5)

Epoch 001 | Loss: 1.4255
Epoch 011 | Loss: 0.6688
Epoch 021 | Loss: 0.3413
Epoch 031 | Loss: 0.2825
Epoch 041 | Loss: 0.2410
Epoch 051 | Loss: 0.2136
Epoch 061 | Loss: 0.1931
Epoch 071 | Loss: 0.1713
Epoch 081 | Loss: 0.1603
Epoch 091 | Loss: 0.1456
Epoch 101 | Loss: 0.1377
Epoch 111 | Loss: 0.1399
Epoch 121 | Loss: 0.1215
Epoch 131 | Loss: 0.1156
Epoch 141 | Loss: 0.1115
Epoch 151 | Loss: 0.1077
Epoch 161 | Loss: 0.0995
Epoch 171 | Loss: 0.0988
Epoch 181 | Loss: 0.0983
Epoch 191 | Loss: 0.0862
Epoch 201 | Loss: 0.0817
Epoch 211 | Loss: 0.0788
Epoch 221 | Loss: 0.0753
Epoch 231 | Loss: 0.0741
Epoch 241 | Loss: 0.0690
Epoch 251 | Loss: 0.0610
Epoch 261 | Loss: 0.0673
Epoch 271 | Loss: 0.0683
Epoch 281 | Loss: 0.0564
Epoch 291 | Loss: 0.0617
Epoch 300 | Loss: 0.0511


In [26]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_5.copy()
num_cluster_5 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_5,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [27]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [28]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_100"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.4305
NMI: 0.6612
HOM: 0.6581


### Data 6

In [29]:
dgi6 = DGI(
    in_omics1=data_6.x_omics1.shape[1],
    in_omics2=data_6.x_omics2.shape[1]
)

dgi6, loss_hist6 = train_dgi(dgi6, data_6, epochs=300)

z = get_embedding_dgi(dgi6, data_6)

Epoch 001 | Loss: 1.8152
Epoch 011 | Loss: 0.9974
Epoch 021 | Loss: 0.6385
Epoch 031 | Loss: 0.4314
Epoch 041 | Loss: 0.3328
Epoch 051 | Loss: 0.2898
Epoch 061 | Loss: 0.2577
Epoch 071 | Loss: 0.2402
Epoch 081 | Loss: 0.2263
Epoch 091 | Loss: 0.2149
Epoch 101 | Loss: 0.1986
Epoch 111 | Loss: 0.1975
Epoch 121 | Loss: 0.1889
Epoch 131 | Loss: 0.1788
Epoch 141 | Loss: 0.1668
Epoch 151 | Loss: 0.1583
Epoch 161 | Loss: 0.1500
Epoch 171 | Loss: 0.1375
Epoch 181 | Loss: 0.1398
Epoch 191 | Loss: 0.1282
Epoch 201 | Loss: 0.1331
Epoch 211 | Loss: 0.1160
Epoch 221 | Loss: 0.1121
Epoch 231 | Loss: 0.1106
Epoch 241 | Loss: 0.0979
Epoch 251 | Loss: 0.1008
Epoch 261 | Loss: 0.0986
Epoch 271 | Loss: 0.0875
Epoch 281 | Loss: 0.0952
Epoch 291 | Loss: 0.0835
Epoch 300 | Loss: 0.0901


In [30]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_6.copy()
num_cluster_6 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_6,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=15, random_state=42)

In [31]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [32]:
dataset = "Amplify_CBMCs_nCells_10075_nGenes_1000"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3944
NMI: 0.6249
HOM: 0.6236


### Data R1

In [26]:
dgir1 = DGI(
    in_omics1=data_r1.x_omics1.shape[1],
    in_omics2=data_r1.x_omics2.shape[1]
)

dgir1, loss_histr1 = train_dgi(dgir1, data_r1, epochs=300)

z = get_embedding_dgi(dgir1, data_r1)

Epoch 001 | Loss: 1.4181
Epoch 011 | Loss: 0.8521
Epoch 021 | Loss: 0.3194
Epoch 031 | Loss: 0.1352
Epoch 041 | Loss: 0.0639
Epoch 051 | Loss: 0.0420
Epoch 061 | Loss: 0.0293
Epoch 071 | Loss: 0.0192
Epoch 081 | Loss: 0.0207
Epoch 091 | Loss: 0.0150
Epoch 101 | Loss: 0.0191
Epoch 111 | Loss: 0.0134
Epoch 121 | Loss: 0.0107
Epoch 131 | Loss: 0.0099
Epoch 141 | Loss: 0.0089
Epoch 151 | Loss: 0.0112
Epoch 161 | Loss: 0.0103
Epoch 171 | Loss: 0.0107
Epoch 181 | Loss: 0.0064
Epoch 191 | Loss: 0.0054
Epoch 201 | Loss: 0.0082
Epoch 211 | Loss: 0.0047
Epoch 221 | Loss: 0.0040
Epoch 231 | Loss: 0.0045
Epoch 241 | Loss: 0.0060
Epoch 251 | Loss: 0.0038
Epoch 261 | Loss: 0.0042
Epoch 271 | Loss: 0.0039
Epoch 281 | Loss: 0.0046
Epoch 291 | Loss: 0.0033
Epoch 300 | Loss: 0.0049


In [27]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r1.copy()
num_cluster_r1 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r1,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

/home/hanle/miniconda3/envs/SpatialGlue/lib/python3.8/site-packages/sklearn/mixture/_base.py:286: ConvergenceWarning: Initialization 1 did not converge. Try different init parameters, or increase max_iter, tol or check for degenerate data.
  warnings.warn(


BayesianGaussianMixture(n_components=9, random_state=42)

In [28]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [29]:
dataset = "ATAC_RNA_Seq_MouseBrain"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.3735
NMI: 0.5495
HOM: 0.5368


### Data R2

In [89]:
dgir2 = DGI(
    in_omics1=data_r2.x_omics1.shape[1],
    in_omics2=data_r2.x_omics2.shape[1]
)

dgir2, loss_histr2 = train_dgi(dgir2, data_r2, epochs=300)

z = get_embedding_dgi(dgir2, data_r2)

Epoch 001 | Loss: 1.6781
Epoch 011 | Loss: 0.6994
Epoch 021 | Loss: 0.3351
Epoch 031 | Loss: 0.1844
Epoch 041 | Loss: 0.1075
Epoch 051 | Loss: 0.0562
Epoch 061 | Loss: 0.0527
Epoch 071 | Loss: 0.0352
Epoch 081 | Loss: 0.0286
Epoch 091 | Loss: 0.0196
Epoch 101 | Loss: 0.0175
Epoch 111 | Loss: 0.0218
Epoch 121 | Loss: 0.0364
Epoch 131 | Loss: 0.0265
Epoch 141 | Loss: 0.0107
Epoch 151 | Loss: 0.0144
Epoch 161 | Loss: 0.0140
Epoch 171 | Loss: 0.0149
Epoch 181 | Loss: 0.0144
Epoch 191 | Loss: 0.0096
Epoch 201 | Loss: 0.0080
Epoch 211 | Loss: 0.0096
Epoch 221 | Loss: 0.0052
Epoch 231 | Loss: 0.0056
Epoch 241 | Loss: 0.0081
Epoch 251 | Loss: 0.0084
Epoch 261 | Loss: 0.0052
Epoch 271 | Loss: 0.0041
Epoch 281 | Loss: 0.0035
Epoch 291 | Loss: 0.0085
Epoch 300 | Loss: 0.0059


In [90]:
# Convert the embedding to a DataFrame for clustering
z_np = z.detach().cpu().numpy()
df_z = pd.DataFrame(z_np)

# Define the number of clusters based on the true labels
adata = adata_rna_r2.copy()
num_cluster_r2 = len(adata.obs['label'].unique())

# Fit the Bayesian Gaussian Mixture model
bayes = BayesianGaussianMixture(n_components = num_cluster_r2,
                                #covariance_type = covariance_type, 
                                random_state = 42,
                                #init_params = '', 
                                #n_init = 1, 
                                #max_iter = 1000
                               )
bayes.fit(df_z)

BayesianGaussianMixture(n_components=7, random_state=42)

In [91]:
# Predict cluster labels
cluster_bayes = bayes.predict(df_z)
pred = cKD_refine_label(np.array(adata.obsm['spatial']), cluster_bayes, k = 6)

In [92]:
dataset = "MISAR_seq_mouse_E15_brain_data"

# True labels
labels_true = adata.obs['label'].values   # numpy array

# Predicted cluster labels
labels_pred = pred 

# Compute ARI and NMI
ARI = adjusted_rand_score(labels_true, labels_pred)
NMI = normalized_mutual_info_score(labels_true, labels_pred)
HOM = homogeneity_score(labels_true, labels_pred)

print(f"ARI: {ARI:.4f}")
print(f"NMI: {NMI:.4f}")
print(f"HOM: {HOM:.4f}")

# Save to a csv file
method = "DGI"
score = []
score.append([method,dataset, ARI, NMI, HOM])
score_df = pd.DataFrame(score, columns = ['Method','DataName', 'ARI', 'NMI', 'HOM'])
output_file = "compare_model.csv"
    #Check if the file exists
if os.path.exists(output_file):
    # Append without writing header
    score_df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # First time: write header
    score_df.to_csv(output_file, mode='w', header=True, index=False)

ARI: 0.2215
NMI: 0.4375
HOM: 0.4210
